# From NumPy to Pandas: Student Achievement, Equity & Learning Outcomes

**A practical Education-sector project that systematically applies the full NumPy chapter and then Pandas**, following the structure of Jake VanderPlas’ *Python Data Science Handbook*.

**NumPy coverage (Chapter 2):**  
02.01 Data types → 02.02 Array basics → 02.03 Ufuncs → 02.04 Aggregates → 02.05 Broadcasting → 02.06 Boolean masks → 02.07 Fancy indexing → 02.08 Sorting → 02.09 Structured arrays

**Pandas coverage:** Objects, indexing, operations, missing data, hierarchical indexing, merge, groupby, pivot tables, time series, and performance tools.

### Real-World Education Scenario
We analyze a mid-sized school district / network of schools with:

- Student-level standardized test scores (Math, Reading, Science)
- Attendance and demographic information
- Multiple schools and grade levels
- Several academic years

**Core questions we answer:**
- What is the overall achievement profile and where are the gaps?
- Which students are at academic risk?
- How large are the equity gaps across demographic groups?
- How do schools compare after adjusting for student composition?
- How have outcomes evolved over time?

> Emphasis throughout: every NumPy and Pandas technique is used to produce **actionable educational insights**.


## 1. Environment Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 14)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13

print(f"NumPy  version : {np.__version__}")
print(f"Pandas version : {pd.__version__}")
print("Environment ready for Education analytics.")


---
# PART I — NumPy Foundations Applied to Student Scores
We begin exactly where the handbook’s NumPy chapter begins: understanding arrays and data types in the context of educational measurement.


## 2. Understanding Data Types & Creating NumPy Arrays

Educational data often mixes integers (scores, days present), floats (GPA, percentages), and categorical codes.  
NumPy forces us to be explicit about dtypes — a valuable discipline.


In [ ]:
rng = np.random.default_rng(42)

# Simulate 500 students × 3 subjects (Math, Reading, Science)
n_students = 500
subjects = ['Math', 'Reading', 'Science']

# Raw scores (0-100) – different means and spreads per subject
math_scores    = rng.normal(72, 14, n_students).clip(20, 100)
reading_scores = rng.normal(75, 12, n_students).clip(25, 100)
science_scores = rng.normal(70, 15, n_students).clip(15, 100)

# Stack into a 2-D array: rows = students, columns = subjects
scores = np.column_stack([math_scores, reading_scores, science_scores])
print("Scores array shape:", scores.shape)
print("dtype:", scores.dtype)
print("\nFirst 5 students (Math, Reading, Science):")
print(scores[:5].round(1))

# Explicit dtype control (common when scores come from integer tests)
scores_int = scores.astype(np.int16)
print("\nMemory footprint comparison:")
print(f"  float64 : {scores.nbytes / 1024:.1f} KB")
print(f"  int16   : {scores_int.nbytes / 1024:.1f} KB")


## 3. The Basics of NumPy Arrays – Indexing, Slicing, Reshaping

We treat the score matrix as the fundamental educational data structure.


In [ ]:
# Basic attributes
print("ndim  :", scores.ndim)
print("shape :", scores.shape)
print("size  :", scores.size)

# Column views (very common in education analytics)
math    = scores[:, 0]
reading = scores[:, 1]
science = scores[:, 2]

print("\nMath scores – first 8 students:")
print(math[:8].round(1))

# Row slicing – a particular classroom or cohort
classroom_a = scores[0:30]          # first 30 students
print("\nClassroom A mean by subject:")
print(classroom_a.mean(axis=0).round(1))

# Reshape example – view as 10 groups of 50 students
grouped = scores.reshape(10, 50, 3)
print("\nReshaped to (10 groups × 50 students × 3 subjects):", grouped.shape)
print("Mean of group 3:", grouped[2].mean(axis=0).round(1))


## 4. Computation on Arrays: Universal Functions (ufuncs)

Ufuncs let us apply mathematical transformations to entire score vectors at once —  
essential for scaling, curving, or converting to different metrics.


In [ ]:
# 4.1 Simple scaling / curving (add 3 points, then clip at 100)
curved = np.clip(scores + 3, 0, 100)
print("Original vs curved (first student):")
print("  Original:", scores[0].round(1))
print("  Curved  :", curved[0].round(1))

# 4.2 Convert to z-scores within each subject (standardization)
means = scores.mean(axis=0)
stds  = scores.std(axis=0)
z_scores = (scores - means) / stds          # broadcasting happens here (see next section)

print("\nSubject means :", means.round(1))
print("Subject stds  :", stds.round(1))
print("Z-scores first 3 students:\n", z_scores[:3].round(2))

# 4.3 Nonlinear transform – example of a growth percentile style mapping
# (illustrative logistic-style compression of extreme scores)
def soft_clip(x, lo=30, hi=95):
    return lo + (hi - lo) * (1 / (1 + np.exp(-(x - 70) / 10)))

adjusted = soft_clip(scores)
print("\nSoft-clipped scores (first student):", adjusted[0].round(1))


## 5. Aggregations – Summarizing Student Performance

Aggregates are the backbone of school report cards and accountability metrics.


In [ ]:
# Overall and per-subject statistics
print("=== District-wide Summary ===")
print(f"Overall mean score     : {scores.mean():.1f}")
print(f"Overall std            : {scores.std():.1f}")
print(f"Min / Max              : {scores.min():.1f} / {scores.max():.1f}")

print("\nPer-subject means :", scores.mean(axis=0).round(1))
print("Per-subject stds  :", scores.std(axis=0).round(1))
print("Per-subject medians:", np.median(scores, axis=0).round(1))

# Percentiles – critical for understanding distribution and setting cut-scores
percentiles = np.percentile(scores, [10, 25, 50, 75, 90], axis=0)
print("\nPercentiles (10/25/50/75/90) by subject:")
print(pd.DataFrame(percentiles, 
                   index=['P10','P25','P50','P75','P90'],
                   columns=subjects).round(1))

# Student-level total and average
student_avg = scores.mean(axis=1)
print(f"\nStudents with average ≥ 85 (high achievers): {(student_avg >= 85).sum()}")
print(f"Students with average < 50 (intensive support): {(student_avg < 50).sum()}")


## 6. Broadcasting – Efficient Operations Without Explicit Loops

Broadcasting is extremely useful when we want to adjust every student’s scores  
by school-level or subject-level effects without writing Python loops.


In [ ]:
# Example 1: Subtract subject means (already done for z-scores)
centered = scores - scores.mean(axis=0)          # (500,3) - (3,) → broadcasts
print("Centered scores shape still:", centered.shape)

# Example 2: Apply different school-level offsets
# Suppose we have 5 schools, each with 100 students, and known school effects
school_effects = np.array([2.5, -1.0, 0.8, -2.2, 1.5])   # shape (5,)
# Expand to student level
school_id = np.repeat(np.arange(5), 100)                 # 0,0,..0,1,1,..1,...
student_school_effect = school_effects[school_id]        # shape (500,)

# Add the school effect to every subject (broadcast across columns)
adjusted_by_school = scores + student_school_effect[:, np.newaxis]
print("\nSchool-adjusted scores (first 3 students of school 0):")
print(adjusted_by_school[:3].round(1))

# Example 3: Create a simple value-added style residual
# residual = actual - expected (where expected could come from a prior model)
expected = 65 + 0.3 * (math_scores - 70)                 # toy prior
math_residual = math_scores - expected
print(f"\nMean Math residual (should be near 0): {math_residual.mean():.2f}")


## 7. Boolean Arrays & Masks – Identifying At-Risk and High-Performing Students

Boolean masking is the most common way educators filter students for intervention or enrichment.


In [ ]:
# Define risk criteria
low_math     = math < 55
low_reading  = reading < 55
low_science  = science < 55
low_overall  = student_avg < 55

# Compound conditions
at_risk = low_overall | ((low_math & low_reading))   # overall low OR weak in both literacy & numeracy
high_achiever = (student_avg >= 88) & (scores.min(axis=1) >= 75)  # high average and no weak subject

print(f"At-risk students          : {at_risk.sum()} ({at_risk.mean()*100:.1f}%)")
print(f"High-achiever students    : {high_achiever.sum()} ({high_achiever.mean()*100:.1f}%)")

# Masking to compute conditional statistics
print(f"\nMean Math score of at-risk students     : {math[at_risk].mean():.1f}")
print(f"Mean Math score of high-achievers       : {math[high_achiever].mean():.1f}")

# Count how many students are weak in each subject
print("\nStudents below 55 by subject:")
print("  Math   :", low_math.sum())
print("  Reading:", low_reading.sum())
print("  Science:", low_science.sum())


## 8. Fancy Indexing – Selecting Specific Cohorts and Rankings


In [ ]:
# Select the 10 highest overall performers
top10_idx = np.argsort(student_avg)[-10:][::-1]
print("Top 10 students (indices and averages):")
print("Indices :", top10_idx)
print("Averages:", student_avg[top10_idx].round(1))
print("Their full score profiles:\n", scores[top10_idx].round(1))

# Select a specific set of students (e.g., a special program cohort)
program_ids = np.array([3, 17, 42, 88, 105, 210, 333, 401])
print("\nSpecial program cohort scores:")
print(scores[program_ids].round(1))

# Combined indexing – rows and columns
# “Give me Math and Science scores of the top 5 students”
print("\nMath & Science of top 5:")
print(scores[top10_idx[:5]][:, [0, 2]].round(1))


## 9. Sorting – Rankings and Ordered Views


In [ ]:
# Sort students by overall average (ascending)
sorted_idx = np.argsort(student_avg)
print("Lowest 5 averages :", student_avg[sorted_idx[:5]].round(1))
print("Highest 5 averages:", student_avg[sorted_idx[-5:]].round(1))

# Rank of every student (1 = best)
ranks = np.empty_like(student_avg, dtype=int)
ranks[sorted_idx[::-1]] = np.arange(1, n_students + 1)
print("\nRanks of first 8 students:", ranks[:8])

# Sort the score matrix itself by Math performance
math_order = np.argsort(math)
scores_sorted_by_math = scores[math_order]
print("\nFirst 5 rows after sorting by Math:")
print(scores_sorted_by_math[:5].round(1))


## 10. Structured Data in NumPy

When we need mixed types (ID, scores, flags) before moving to Pandas,  
NumPy structured arrays provide a lightweight tabular structure.


In [ ]:
# Define a structured dtype for a compact student record
student_dtype = np.dtype([
    ('id',       'i4'),
    ('math',     'f4'),
    ('reading',  'f4'),
    ('science',  'f4'),
    ('at_risk',  '?'),
    ('school_id','i2')
])

structured = np.zeros(n_students, dtype=student_dtype)
structured['id']        = np.arange(1001, 1001 + n_students)
structured['math']      = math
structured['reading']   = reading
structured['science']   = science
structured['at_risk']   = at_risk
structured['school_id'] = school_id

print("Structured array sample (first 4 records):")
print(structured[:4])

print("\nMean Math of at-risk students via structured field:")
print(structured[structured['at_risk']]['math'].mean().round(1))


---
# PART II — Moving to Pandas for Richer Educational Analysis

NumPy gives us speed and low-level control.  
Pandas adds labels, grouping, merging, time-series tools, and far more convenient handling of mixed-type educational data.


## 11. Building a Student-Level DataFrame


In [ ]:
# Create a clean student DataFrame from the NumPy arrays
students = pd.DataFrame({
    'student_id': np.arange(1001, 1001 + n_students),
    'school_id': school_id,
    'math': math.round(1),
    'reading': reading.round(1),
    'science': science.round(1),
    'avg_score': student_avg.round(1),
    'at_risk': at_risk,
    'high_achiever': high_achiever
})

# Add synthetic demographics for equity analysis
students['gender'] = rng.choice(['F', 'M'], n_students)
students['ell'] = rng.choice([False, True], n_students, p=[0.82, 0.18])          # English Language Learner
students['sped'] = rng.choice([False, True], n_students, p=[0.88, 0.12])         # Special Education
students['frpl'] = rng.choice([False, True], n_students, p=[0.55, 0.45])         # Free/Reduced Price Lunch (SES proxy)

print("Student DataFrame:")
display(students.head(8))
print(f"\nShape: {students.shape}")
print(f"At-risk rate: {students['at_risk'].mean()*100:.1f}%")


## 12. Indexing, Selection & Vectorized Operations in Pandas


In [ ]:
# Classic educational filters
print("High-achieving ELL students:")
display(students.query("high_achiever and ell")[['student_id', 'math', 'reading', 'science', 'avg_score']])

# Create a simple proficiency flag (example cut-score = 70)
for subj in ['math', 'reading', 'science']:
    students[f'{subj}_prof'] = students[subj] >= 70

print("\nProficiency rates:")
print(students[['math_prof', 'reading_prof', 'science_prof']].mean().mul(100).round(1).astype(str) + '%')

# Real-world derived metric: “balanced achiever” (proficient in all three)
students['balanced'] = students['math_prof'] & students['reading_prof'] & students['science_prof']
print(f"\nBalanced proficient students: {students['balanced'].mean()*100:.1f}%")


## 13. Handling Missing Data in Educational Records

Attendance systems, transfer students, and incomplete test administrations create missing values.


In [ ]:
# Introduce realistic missingness
students_miss = students.copy()
miss_idx = rng.choice(students_miss.index, size=35, replace=False)
students_miss.loc[miss_idx, 'science'] = np.nan

print(f"Missing Science scores: {students_miss['science'].isna().sum()}")

# Strategies used in practice
# 1. Flag and exclude from certain aggregates
valid_science = students_miss.dropna(subset=['science'])
print(f"Mean Science (complete cases): {valid_science['science'].mean():.1f}")

# 2. Impute with school mean (simple but common)
school_means = students_miss.groupby('school_id')['science'].transform('mean')
students_miss['science_imputed'] = students_miss['science'].fillna(school_means)
print(f"Mean Science after school-mean imputation: {students_miss['science_imputed'].mean():.1f}")


## 14. GroupBy & Hierarchical Views – School and Equity Analysis


In [ ]:
# School-level summary
school_summary = students.groupby('school_id').agg(
    n_students=('student_id', 'count'),
    mean_avg=('avg_score', 'mean'),
    math_mean=('math', 'mean'),
    reading_mean=('reading', 'mean'),
    science_mean=('science', 'mean'),
    at_risk_rate=('at_risk', 'mean'),
    frpl_rate=('frpl', 'mean')
).round(2)

print("School-level performance summary:")
display(school_summary)

# Equity analysis – achievement by FRPL (socioeconomic proxy)
print("\n=== Equity Lens: Performance by FRPL status ===")
equity = students.groupby('frpl')[['math', 'reading', 'science', 'avg_score']].mean().round(1)
equity.index = ['Non-FRPL', 'FRPL']
display(equity)

gap = equity.loc['Non-FRPL'] - equity.loc['FRPL']
print("\nAchievement gaps (Non-FRPL − FRPL):")
print(gap.round(1))

# Visual
equity.T.plot(kind='bar', figsize=(9, 4), title='Mean Scores by FRPL Status')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(title='')
plt.tight_layout()
plt.show()


## 15. Pivot Tables – Multi-dimensional Education Reports


In [ ]:
# Classic report: Mean score by School × Subject
# First melt to long form
long = students.melt(
    id_vars=['student_id', 'school_id', 'frpl', 'ell', 'sped'],
    value_vars=['math', 'reading', 'science'],
    var_name='subject', value_name='score'
)

pivot_school_subj = long.pivot_table(
    values='score',
    index='school_id',
    columns='subject',
    aggfunc='mean'
).round(1)

print("Mean score by School × Subject:")
display(pivot_school_subj)

# Equity pivot: FRPL × Subject
pivot_equity = long.pivot_table(
    values='score',
    index='frpl',
    columns='subject',
    aggfunc=['mean', 'std', 'count']
).round(1)
print("\nDetailed equity view (FRPL × Subject):")
display(pivot_equity)


## 16. Adding a Longitudinal / Time Dimension

Most serious education analysis is longitudinal (cohorts over grades or years).


In [ ]:
# Simulate 4 years of district average scores
years = pd.date_range('2022-06-01', periods=4, freq='YE')
district_trend = pd.DataFrame({
    'math':    [68.5, 70.1, 71.8, 72.4],
    'reading': [71.2, 72.0, 73.5, 74.1],
    'science': [66.8, 67.9, 69.2, 70.0]
}, index=years)

print("District-wide trend (end-of-year averages):")
display(district_trend)

# YoY growth
print("\nYear-over-year growth (points):")
display(district_trend.diff().round(1))

district_trend.plot(marker='o', title='District Average Scores Over Time')
plt.ylabel('Mean Score')
plt.tight_layout()
plt.show()


## 17. Best-Practice Pattern: NumPy for Computation, Pandas for Structure

In production education analytics we often:
1. Keep heavy numeric work in NumPy (fast, memory-efficient)
2. Use Pandas for labels, grouping, merging, and reporting


In [ ]:
# Example: compute subject z-scores with NumPy, then attach back to DataFrame
score_matrix = students[['math', 'reading', 'science']].to_numpy()
z = (score_matrix - score_matrix.mean(axis=0)) / score_matrix.std(axis=0)

students[['math_z', 'reading_z', 'science_z']] = z.round(2)

print("Students with z-scores attached:")
display(students[['student_id', 'math', 'math_z', 'reading', 'reading_z']].head(6))

# Fast NumPy filter → Pandas view
high_z_math_idx = np.where(z[:, 0] > 1.5)[0]
print(f"\nStudents with Math z > 1.5: {len(high_z_math_idx)}")
display(students.iloc[high_z_math_idx][['student_id', 'math', 'math_z', 'school_id']].head())


## 18. Synthesis – Key Educational Insights


In [ ]:
print("=== DISTRICT SNAPSHOT ===")
print(f"Total students analyzed     : {len(students)}")
print(f"Overall mean score          : {students['avg_score'].mean():.1f}")
print(f"At-risk rate                : {students['at_risk'].mean()*100:.1f}%")
print(f"High-achiever rate          : {students['high_achiever'].mean()*100:.1f}%")
print(f"Balanced proficiency rate   : {students['balanced'].mean()*100:.1f}%")

print("\n=== EQUITY GAPS (Non-FRPL − FRPL) ===")
print(gap.round(1).to_string())

print("\n=== SCHOOL VARIATION ===")
print(f"Highest school mean avg     : {school_summary['mean_avg'].max():.1f} (School {school_summary['mean_avg'].idxmax()})")
print(f"Lowest  school mean avg     : {school_summary['mean_avg'].min():.1f} (School {school_summary['mean_avg'].idxmin()})")
print(f"Range across schools        : {school_summary['mean_avg'].max() - school_summary['mean_avg'].min():.1f} points")

# Final visual panel
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Score distributions
students[['math', 'reading', 'science']].plot.hist(bins=20, alpha=0.6, ax=axes[0,0])
axes[0,0].set_title('Score Distributions by Subject')
axes[0,0].set_xlabel('Score')

# School means
school_summary['mean_avg'].plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Mean Average Score by School')
axes[0,1].set_ylabel('Score')
axes[0,1].tick_params(axis='x', rotation=0)

# Equity gaps
gap.plot(kind='barh', ax=axes[1,0], color='crimson')
axes[1,0].set_title('Achievement Gaps (Non-FRPL − FRPL)')
axes[1,0].set_xlabel('Score points')

# At-risk rate by school
school_summary['at_risk_rate'].mul(100).plot(kind='bar', ax=axes[1,1], color='darkorange')
axes[1,1].set_title('At-Risk Rate by School (%)')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(
"=== EXECUTIVE INSIGHTS FOR EDUCATION LEADERS ===\n"
"1. Clear achievement gaps exist by socioeconomic status (FRPL).\n"
"2. School-level variation is substantial - some schools outperform others\n"
"   even before any formal value-added adjustment.\n"
"3. A non-trivial share of students meet at-risk criteria and need\n"
"   targeted academic and attendance interventions.\n"
"4. Science shows the widest spread and lowest mean - a potential\n"
"   curriculum or instructional focus area.\n"
"5. Combining NumPy (fast numeric work) with Pandas (grouping, labeling,\n"
"   reporting) gives both performance and clarity for education analytics."
)


## 19. Mapping to the Python Data Science Handbook

### NumPy (Chapter 2)
| Handbook Section | Technique | Education Application |
|------------------|-----------|-----------------------|
| 02.01 Data types | dtype control, memory | Efficient storage of large score matrices |
| 02.02 Array basics | shape, indexing, reshape | Student × Subject matrices, cohort views |
| 02.03 Ufuncs | vectorized math | Curving, scaling, soft-clipping scores |
| 02.04 Aggregates | mean, std, percentile | District & school report-card metrics |
| 02.05 Broadcasting | implicit expansion | School-effect adjustment, centering, z-scores |
| 02.06 Boolean masks | conditional selection | At-risk & high-achiever identification |
| 02.07 Fancy indexing | integer array indexing | Top performers, program cohorts |
| 02.08 Sorting | argsort, ranking | Student & school rankings |
| 02.09 Structured arrays | mixed-type records | Lightweight student records before Pandas |

### Pandas (Chapter 3)
| Technique | Education Application |
|-----------|-----------------------|
| DataFrame creation & indexing | Student-level analytic file |
| Operations & derived columns | Proficiency flags, balanced achiever |
| Missing data handling | Incomplete test administrations |
| GroupBy & aggregation | School summaries, equity gaps |
| Pivot tables | School × Subject, FRPL × Subject reports |
| Time series | Multi-year district trends |
| query / eval | Fast filtering of complex student groups |

---

### Next Steps for Real Education Analytics
1. Replace synthetic scores with real assessment data (state tests, MAP, i-Ready, etc.).
2. Add growth metrics (e.g., student growth percentiles) and prior-year scores.
3. Incorporate attendance, behavior, and course-taking patterns.
4. Move toward value-added or hierarchical linear models for school/teacher effects.
5. Build automated equity dashboards that update with each new assessment window.

**You now have a single notebook that walks through the entire NumPy chapter and the core of the Pandas chapter, all grounded in a realistic Education problem.**


## 20. Further Resources

- Jake VanderPlas – [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/)
  - [Chapter 2 – NumPy](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html)
  - [Chapter 3 – Pandas](https://jakevdp.github.io/PythonDataScienceHandbook/03.00-introduction-to-pandas.html)
- Education data sources: state education agencies, NCES, EdFacts, district data warehouses, assessment vendors
- Domain concepts: student growth percentiles, proficiency gaps, value-added modeling, multi-tiered systems of support (MTSS)

---

*Notebook generated for practical mastery of NumPy + Pandas in the Education domain.*  
*All data is synthetic but statistically realistic for teaching purposes.*
